# M04 — Joins y KPIs

[← Anterior](../M03-transformacion-datos/03-lab-reglas-negocio.ipynb) · [Siguiente →](02-lab-joins.ipynb)

Un número de negocio mentiroso casi siempre viene de **mezclar dos tamaños de fila**, no de un `sum` mal escrito.

Antes de cruzar nada, fíjate en esto (como un ticket de supermercado):

- Un **pedido** (`order_id`) es el ticket entero: “O1”.
- Una **línea** es un producto dentro del ticket: “100 € de auriculares” y “50 € de cable” pueden ser **el mismo** pedido.
- Un **cliente** vive en otra lista. Si la línea apunta a un id que no está en esa lista, es un **huérfano** (en NovaShop, los `CX*`).

En las celdas de abajo el juguete es pequeño a propósito: **un pedido con dos líneas**, otro pedido de una línea, y un huérfano. Así se ve la diferencia. Luego el lab usa NovaShop entero.

Ejecuta las celdas **aquí**, en este mismo fichero (clase, juntos). Va **montado**: explicación + código + lo que tienes que ver. Lo que construyes tú está en el **lab**.

Kernel: **Python (NovaShop)**.


## Arranque

La primera celda **no es Spark todavía**: busca la raíz del repo (aunque este notebook no esté en la carpeta de arriba) y deja `RAW`, `STAGING` y `CURATED` listos. La segunda pide una `SparkSession` en `local[*]` (todos los cores de esta máquina; no hay clúster).

Al ejecutar: rutas impresas y una versión `3.5.x` con master `local[*]`.


In [ ]:
import sys
from pathlib import Path

# El notebook puede estar en trabajo/; subimos hasta encontrar el repo.
_here = Path.cwd().resolve()
ROOT = next(
    p
    for p in [_here, *_here.parents]
    if (p / "labs" / "_shared" / "session.py").is_file()
)
sys.path.insert(0, str(ROOT / "labs" / "_shared"))

from paths import RAW, STAGING, CURATED  # rutas absolutas, no Path("data/raw")
from session import get_spark

print("ROOT   ", ROOT)
print("RAW    ", RAW, "existe:", RAW.is_dir())
print("STAGING", STAGING)
print("CURATED", CURATED)


In [ ]:
# getOrCreate: si ya hay sesión en este kernel, la reusa (mismo puerto 4040)
spark = get_spark('novashop-clase-m04')
print(spark.version, spark.sparkContext.master)


## Las dos tablas, en bruto

`clientes` tiene **una fila por persona**. `lineas` tiene **una fila por producto vendido**, no por pedido.

Al ejecutar verás 2 clientes y **4 líneas**. `O1` aparece **dos veces** (100 € y 50 €): eso no son dos ventas de compañía, es **un** ticket con dos productos. `O3` / `CX9` no tiene ficha en `clientes`.


In [ ]:
from pyspark.sql import Row
from pyspark.sql.functions import col, sum as fsum, countDistinct, avg

clientes = spark.createDataFrame([
    Row(customer_id="C1", country="ES"),
    Row(customer_id="C2", country="FR"),  # no compra en este juguete; da igual
])
# gmv_line = dinero de ESA línea (un producto), no del pedido entero
lineas = spark.createDataFrame([
    Row(order_id="O1", customer_id="C1", gmv_line=100.0, is_billable=True),
    Row(order_id="O1", customer_id="C1", gmv_line=50.0, is_billable=True),   # mismo pedido
    Row(order_id="O2", customer_id="C1", gmv_line=30.0, is_billable=True),
    Row(order_id="O3", customer_id="CX9", gmv_line=999.0, is_billable=True),  # huérfano
])
print("clientes (1 fila = 1 persona)")
clientes.show()
print("lineas (1 fila = 1 producto; O1 está dos veces)")
lineas.show()


## Cruzar: “¿esta línea tiene cliente de verdad?”

Un join no suma dinero. Solo pregunta, **línea a línea**: ¿el `customer_id` está en `clientes`?

- **inner** — me quedo solo si hay emparejamiento. `CX9` (y sus 999 €) **desaparecen**. Cuenta: **3** (las dos de `O1` y la de `O2`).
- **left** — me quedo con **todas** las líneas. `CX9` sigue, `country` sale `null`. Cuenta: **4**. Sirve para *ver* cuánto se cae, no para decir “vendimos 999 a un cliente”.
- **left_anti** — “líneas cuyo cliente **no** está en la lista”. Es la foto del huérfano. Verás `O3`.

Al ejecutar: inner **3**, left **4**, anti = `O3`.


In [ ]:
print("inner (fuera el huérfano)", lineas.join(clientes, "customer_id", "inner").count())
print("left  (siguen las 4)     ", lineas.join(clientes, "customer_id", "left").count())
print("quién no está en clientes:")
lineas.join(clientes, "customer_id", "left_anti").show()


## `sales` no es “las ventas del pedido”

En los labs llamamos `sales` a: líneas **cobrables** (`is_billable`) **con cliente conocido** (inner).

Eso **no** agrupa. Sigue habiendo **una fila por producto**. `O1` sigue saliendo dos veces. El nombre engaña: no es un ticket cerrado, es el recorte “estas líneas sí cuentan para dinero atribuible”.

Al ejecutar: **3** filas, GMV de línea 100 / 50 / 30. El 999 ya no está.


In [ ]:
# Recorte, no agregación: mismas 3 líneas, ahora con country
sales = lineas.join(clientes, "customer_id", "inner").where(col("is_billable"))
print("filas en sales (sigue siendo grano LÍNEA):", sales.count())
sales.show()


## El ticket medio: no hagas la media de las filas

Pregunta de negocio: “¿cuánto deja de media **un pedido**?”

En `sales` hay **3 productos** y solo **2 tickets** (`O1` = 150 €, `O2` = 30 €).

| Cálculo | Qué está promediando | Número |
|---------|----------------------|-------:|
| `avg(gmv_line)` | las **3 líneas** (100, 50, 30) | **60** ← mentira útil |
| `sum(gmv_line) / countDistinct(order_id)` | los **2 pedidos** (150 y 30) | **90** ← ticket medio |

`avg` no sabe qué filas son el mismo `order_id`. Por eso la regla del curso es siempre:

`GMV = sum(gmv_line)` y `pedidos = countDistinct(order_id)` y `AOV = GMV / pedidos`.

El `groupBy("country")` no cambia de grano: es **el mismo 180 €** partido por país (aquí todo es ES).

Al ejecutar: una fila `gmv=180`, `orders=2`, `aov=90`; `avg_linea=60`; país ES = 180.


In [ ]:
# Lo que NO hay que usar para el ticket:
sales.agg(avg("gmv_line").alias("avg_linea")).show()  # 60: media de productos

# Lo que SÍ: dinero total y cuántos tickets distintos
kpis = sales.agg(
    fsum("gmv_line").alias("gmv"),                 # 180
    countDistinct("order_id").alias("orders"),     # 2  (no 3)
)
kpis = kpis.withColumn("aov", col("gmv") / col("orders"))  # 90
kpis.show()

# Mismo 180, cortado por país (sigue siendo suma de líneas)
sales.groupBy("country").agg(fsum("gmv_line").alias("gmv")).show()


En NovaShop pasa lo mismo a lo grande: `sales` son miles de **líneas**; el AOV divide por pedidos distintos, no por `count()` de filas.

**Siguiente:** [lab de joins](02-lab-joins.ipynb) sobre el dataset real.
